# SearchAI Index Server + GPU FastEmbed + Local Qdrant Benchmark

This notebook is the GPU/self-hosted-comparison leg of
[`local-inference-bench`](https://github.com/inamdarmihir/local-inference-bench),
run in Google Colab instead of locally because:

1. SearchBlox's **SearchAI Index Server** installer is Linux-only. Colab's
   VM is a real Ubuntu Linux host with `sudo`, unlike a Mac, and needs no
   Docker/systemd workaround.
2. Colab gives free (or paid-tier) **GPU** access, letting us test whether
   hardware acceleration changes the CPU-only cost/throughput story from
   the base repo.

**Honesty contract (read before trusting any number below):**

- Every result is tagged `measured`, `cited`, or `projected`, exactly like
  the base repo's `cost_comparison.py` -- these are never blended.
- The GPU model **varies by Colab session/tier** (free T4, or L4/A100 on
  Colab Pro) -- every GPU number is always printed with its `nvidia-smi`
  name inline, never as a bare "GPU" figure.
- Wall-clock times include Colab's own shared-tenancy overhead; this is
  not a clean-room benchmark and repeat runs may vary more than a
  dedicated instance would.
- The Colab runtime is **ephemeral** -- run the final "download artifacts"
  cell before your session recycles, or you lose the results.
- Colab's vCPU is **not** the AWS c7g.xlarge the base repo's cited pricing
  is anchored to. The CPU number measured here is a new, separate data
  point for context, not a substitute for that existing citation.
- SearchAI Index Server's GPU support (or lack of it) is **empirically
  observed** in this notebook (GPU utilization sampling + a CUDA-library
  check on the running process), never assumed from having a GPU runtime
  attached -- no GPU documentation for this product was found anywhere
  during research.
- If the SearchAI Qdrant-dialect listener rejects a plain-text
  (server-embeds-it) upsert, that is reported as an explicit, valid
  result with the exact request/response -- this notebook does not
  silently fall back to a different API to make the leg "succeed."

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better),
then `Runtime > Run all`. Expect ~15-25 minutes total, most of it spent on
the corpus checkout and the SearchAI Index Server download/boot.

## 1. Environment fingerprint

Printed first, unconditionally, so every later number can be footnoted against the actual hardware of this specific session.

In [ ]:
import json
import platform
import subprocess
import sys
import datetime

env = {}
env["python_version"] = sys.version
env["platform"] = platform.platform()
env["in_colab"] = "google.colab" in sys.modules
env["run_started_utc"] = datetime.datetime.utcnow().isoformat() + "Z"

try:
    env["nvidia_smi"] = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    ).stdout.strip()
except FileNotFoundError:
    env["nvidia_smi"] = "NOT FOUND (no GPU runtime attached)"

try:
    env["cpu_info"] = subprocess.run(
        ["lscpu"], capture_output=True, text=True, timeout=10
    ).stdout
except FileNotFoundError:
    env["cpu_info"] = platform.processor() or "unknown"

print(json.dumps({k: v for k, v in env.items() if k != "cpu_info"}, indent=2))
print()
print(env["cpu_info"][:800])

HAS_GPU = bool(env["nvidia_smi"]) and "NOT FOUND" not in env["nvidia_smi"]
if not HAS_GPU:
    print()
    print("=" * 70)
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type >")
    print("GPU, then Runtime > Restart and run all. The GPU FastEmbed leg")
    print("below will be skipped (not faked) if this stays False.")
    print("=" * 70)
print("HAS_GPU =", HAS_GPU)

## 2. Clone the base repo and install its pinned dependencies

Installing `requirements.txt` unmodified first means the CPU baseline leg (section 4) runs with the exact same dependency versions as the rest of the repo.

In [ ]:
!git clone --depth 1 https://github.com/inamdarmihir/local-inference-bench.git /content/local-inference-bench
%cd /content/local-inference-bench
!pip install -q -r requirements.txt

## 3. Write the new helper modules

`corpus.load_corpus(dirs)` scans one directory non-recursively by design
(see its own docstring) -- it is a protected file in the base repo and is
never modified. `multi_dir_corpus.py` composes over its public API to
handle a deeply-nested real corpus. `searchai_client.py` holds the
install/probe/ingest functions for SearchAI Index Server. Both are also
committed as real files in this repo (see the PR); they are rewritten
here too so this notebook stays fully self-contained and runnable even
before that PR merges.

In [ ]:
%%writefile multi_dir_corpus.py
"""
multi_dir_corpus.py

corpus.load_corpus(dirs) scans each directory non-recursively
(directory.glob("*.md")) by design -- see corpus.py's own docstring. A
deeply-nested real-world corpus (e.g. a documentation site with
concepts/, tasks/, tutorials/ subtrees, each holding its own .md files)
needs its leaf directories discovered and passed in individually. This
module does that discovery and fans out to the existing, unmodified
load_corpus() once per leaf directory -- it does not reimplement chunking
and does not modify corpus.py.

Used by notebooks/searchai_colab_benchmark.ipynb to load a large, real
corpus (e.g. kubernetes/website's content/en/docs/ tree) for the GPU/
SearchAI benchmark legs, while still driving run_local_benchmark.py's own
--corpus-dir (repeatable) for the CPU baseline so that leg's methodology
stays identical to the rest of this repo.
"""

import random
from pathlib import Path

from corpus import Chunk, load_corpus


def find_md_leaf_dirs(root: Path) -> list[Path]:
    """Every directory at or under root that directly contains >=1 *.md
    file, sorted for deterministic ordering."""
    root = Path(root)
    dirs = {md_file.parent for md_file in root.rglob("*.md")}
    return sorted(dirs)


def load_corpus_recursive(
    root: Path,
    max_chunks: int | None = None,
    seed: int = 42,
) -> tuple[list[Chunk], list[Path]]:
    """Discovers every leaf directory under root containing markdown
    files and loads/chunks all of them via corpus.load_corpus(), one
    directory at a time (matching its documented non-recursive-per-
    directory contract), then concatenates in leaf-dir order.

    If max_chunks is given and there are more chunks than that, takes a
    reproducible (seeded) random subsample so notebook runtime stays
    predictable regardless of how large the source corpus happens to be
    on a given day.

    Returns (chunks, leaf_dirs) so callers can pass leaf_dirs straight
    into run_local_benchmark.py's own --corpus-dir (repeatable) for a
    methodology-identical CPU baseline over the same directories.
    """
    leaf_dirs = find_md_leaf_dirs(root)
    if not leaf_dirs:
        raise SystemExit(f"No markdown files found anywhere under {root}.")

    all_chunks: list[Chunk] = []
    for directory in leaf_dirs:
        all_chunks.extend(load_corpus([directory]))

    if max_chunks is not None and len(all_chunks) > max_chunks:
        rng = random.Random(seed)
        all_chunks = rng.sample(all_chunks, max_chunks)

    return all_chunks, leaf_dirs


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Recursively discover and load a nested markdown "
        "corpus (e.g. a documentation site) by fanning out to "
        "corpus.load_corpus() once per leaf directory, since that "
        "function only scans one directory non-recursively."
    )
    parser.add_argument("root", type=Path, help="Root directory to search under.")
    parser.add_argument(
        "--max-chunks",
        type=int,
        default=None,
        help="Cap the corpus to this many chunks via a reproducible "
        "random subsample (default: no cap).",
    )
    parser.add_argument(
        "--seed", type=int, default=42, help="Subsample seed (default: 42)."
    )
    args = parser.parse_args()

    chunks, leaf_dirs = load_corpus_recursive(
        args.root, max_chunks=args.max_chunks, seed=args.seed
    )
    total_words = sum(c.word_count for c in chunks)
    print(f"{len(leaf_dirs)} leaf directories with markdown files under {args.root}")
    print(f"{len(chunks)} chunks from {len(set(c.source_file for c in chunks))} files")
    print(f"{total_words} total words")
    if chunks:
        print(f"{total_words / len(chunks):.0f} avg words/chunk")

In [ ]:
%%writefile searchai_client.py
"""
searchai_client.py

Install/run/probe helpers for SearchBlox's SearchAI Index Server, written
as functions (not just notebook cells) so the sequence is reviewable and
reusable outside Colab. Every probe returns the raw HTTP status/body
alongside its verdict -- nothing here infers or assumes a result the
server didn't actually return.

This module shares no code with corpus.py/tokens.py/pricing.py/
run_local_benchmark.py/push_to_qdrant.py: it talks to a separate external
process (the SearchAI Index Server binary) over HTTP, not to fastembed or
an in-process qdrant-client.

Verified facts this module builds on (confirmed by reading the actual
install script and SearchBlox's own documentation, not assumed):
  - The install is Linux-only; default port 9200, default install prefix
    /opt/searchai-index.
  - Two install paths exist: the one-line installer (curl | sudo bash),
    which sets up a systemd service, and a documented manual fallback --
    extract the release tarball and run ./run.sh in the foreground.
    Environments without a full init system (containers, Colab) should
    prefer the foreground path; this module implements that path.
  - The Qdrant dialect listener is off by default. Enabling it means
    appending dialects.qdrant-port=<port> to
    <prefix>/conf/server.properties and restarting.
  - No published documentation shows a plain-text (vector-less) upsert
    body for the Qdrant dialect. Whether the dialect embeds text
    server-side at ingest, versus requiring a client-supplied vector like
    real Qdrant, is the single open question this module exists to answer
    empirically -- see probe_plaintext_upsert().
"""

import json
import re
import subprocess
import threading
import time
import urllib.error
import urllib.request
from dataclasses import dataclass
from pathlib import Path

DEFAULT_VERSION = "1.3.0"
DEFAULT_PREFIX = Path("/opt/searchai-index")
DEFAULT_PORT = 9200
DEFAULT_QDRANT_PORT = 6333
RELEASE_BASE_URL = "https://index-server.searchblox.com"


@dataclass
class ProbeResult:
    name: str
    ok: bool
    status_code: int | None
    request: dict | None
    response_body: str


def download_and_unpack(
    version: str = DEFAULT_VERSION,
    arch: str = "amd64",
    work_dir: Path = Path("/content"),
) -> Path:
    """Downloads and unpacks the same release tarball the official
    one-line installer fetches, without running its systemd install
    step. Returns the unpacked bundle directory."""
    tarball = f"searchai-index-server-{version}-linux-{arch}.tar.gz"
    url = f"{RELEASE_BASE_URL}/releases/{version}/{tarball}"
    dest = work_dir / tarball
    urllib.request.urlretrieve(url, dest)
    subprocess.run(["tar", "-xzf", str(dest), "-C", str(work_dir)], check=True)
    unpacked_name = (
        subprocess.run(
            ["tar", "-tzf", str(dest)], capture_output=True, text=True, check=True
        )
        .stdout.splitlines()[0]
        .split("/")[0]
    )
    return work_dir / unpacked_name


def start_foreground(bundle_dir: Path, log_path: Path) -> subprocess.Popen:
    """Launches <bundle_dir>/run.sh in the background (not systemd),
    logging stdout/stderr to log_path. Colab (and most containers) have
    no full init system to reliably supervise a systemd service, so this
    is the primary launch path; the full installer is only a documented
    fallback attempt."""
    log_file = open(log_path, "w")
    return subprocess.Popen(
        ["./run.sh"], cwd=str(bundle_dir), stdout=log_file, stderr=subprocess.STDOUT
    )


def wait_for_health(
    port: int = DEFAULT_PORT,
    proc: subprocess.Popen | None = None,
    log_path: Path | None = None,
    timeout_s: int = 150,
    interval_s: int = 5,
) -> bool:
    """Polls http://localhost:<port>/ until it responds (status < 500) or
    timeout_s elapses. If proc exits early, or the timeout is hit, prints
    the process's own log tail and returns False -- never hangs silently
    or retries past the stated budget."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if proc is not None and proc.poll() is not None:
            print(f"run.sh exited early with code {proc.returncode}")
            if log_path and log_path.exists():
                print(log_path.read_text()[-4000:])
            return False
        try:
            with urllib.request.urlopen(f"http://localhost:{port}/", timeout=3) as resp:
                if resp.status < 500:
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(f"Timed out after {timeout_s}s waiting for http://localhost:{port}/")
    if log_path and log_path.exists():
        print(log_path.read_text()[-4000:])
    return False


def find_admin_api_key(log_path: Path) -> str | None:
    """A fresh install auto-generates and prints an admin API key (per
    SearchBlox's own published example). Looks for it in the launch log.
    Returns None if not found -- callers should then try unauthenticated
    requests, since auth is documented as off by default until
    server.api-key is set."""
    if not log_path.exists():
        return None
    text = log_path.read_text()
    match = re.search(
        r"(?:api[-_ ]?key)\s*[:=]\s*([A-Za-z0-9_\-]{16,})", text, re.IGNORECASE
    )
    return match.group(1) if match else None


def enable_qdrant_dialect(
    prefix: Path = DEFAULT_PREFIX, qdrant_port: int = DEFAULT_QDRANT_PORT
) -> None:
    """Appends dialects.qdrant-port=<qdrant_port> to server.properties if
    not already present. Caller is responsible for restarting the server
    afterwards -- this alone has no effect on an already-running process."""
    props_path = prefix / "conf" / "server.properties"
    text = props_path.read_text() if props_path.exists() else ""
    if "dialects.qdrant-port" not in text:
        with open(props_path, "a") as f:
            f.write(f"\ndialects.qdrant-port={qdrant_port}\n")


def _request(
    method: str, url: str, body: dict | None = None, api_key: str | None = None
) -> ProbeResult:
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            return ProbeResult(
                name=url,
                ok=resp.status < 400,
                status_code=resp.status,
                request=body,
                response_body=resp.read().decode(errors="replace"),
            )
    except urllib.error.HTTPError as e:
        return ProbeResult(
            name=url,
            ok=False,
            status_code=e.code,
            request=body,
            response_body=e.read().decode(errors="replace"),
        )
    except urllib.error.URLError as e:
        return ProbeResult(
            name=url, ok=False, status_code=None, request=body, response_body=str(e)
        )


def probe_version(port: int = DEFAULT_PORT, api_key: str | None = None) -> ProbeResult:
    return _request("GET", f"http://localhost:{port}/", api_key=api_key)


def probe_create_qdrant_collection(
    collection: str, dim: int, qdrant_port: int = DEFAULT_QDRANT_PORT
) -> ProbeResult:
    """Known-good per SearchBlox's own docs (a client-supplied-vector
    collection create): establishes the Qdrant dialect responds at all,
    before testing the riskier vector-less-upsert case below."""
    body = {"vectors": {"size": dim, "distance": "Cosine"}}
    return _request(
        "PUT", f"http://localhost:{qdrant_port}/collections/{collection}", body=body
    )


def probe_plaintext_upsert(
    collection: str, text: str, qdrant_port: int = DEFAULT_QDRANT_PORT
) -> ProbeResult:
    """THE critical probe. Sends a point with no "vector" key at all --
    only a payload -- to test whether the Qdrant dialect embeds text
    server-side at ingest (the premise the whole SearchAI leg depends on)
    or requires a client-supplied vector like real Qdrant. No published
    documentation shows this working either way; a rejection here is a
    valid, expected, and reportable result -- not a bug to route around."""
    body = {"points": [{"id": 1, "payload": {"body": text}}]}
    return _request(
        "PUT",
        f"http://localhost:{qdrant_port}/collections/{collection}/points",
        body=body,
    )


def probe_opensearch_autoembed(
    index: str,
    text: str,
    model: str = "bge-small-en-v1.5",
    dim: int = 384,
    port: int = DEFAULT_PORT,
) -> tuple[ProbeResult, ProbeResult]:
    """The alternate, independently-documented path: an OpenSearch index
    with a knn_vector field carrying an "embed" block naming the source
    field and model (per SearchBlox's own published mapping example).
    Creates that mapping, then indexes one plain-text document.

    A pass here does NOT validate the Qdrant-dialect premise tested by
    probe_plaintext_upsert() -- it is a different API surface entirely and
    must be reported as its own, separately-labeled result, never
    substituted for the Qdrant-dialect probe."""
    mapping = {
        "mappings": {
            "properties": {
                "body": {"type": "text"},
                "vec": {
                    "type": "knn_vector",
                    "dimension": dim,
                    "space_type": "cosinesimil",
                    "embed": {"source_field": "body", "model": model},
                },
            }
        }
    }
    create = _request("PUT", f"http://localhost:{port}/{index}", body=mapping)
    index_doc = _request(
        "POST", f"http://localhost:{port}/{index}/_doc/1", body={"body": text}
    )
    return create, index_doc


def sample_gpu_utilization(
    stop_event: threading.Event, interval_s: float = 1.0
) -> list[int]:
    """Samples `nvidia-smi --query-gpu=utilization.gpu` once per
    interval_s until stop_event is set. A coarse proxy -- 1Hz sampling can
    miss short bursts -- so pair with process_uses_cuda() below for a
    second, independent signal rather than trusting this alone."""
    samples: list[int] = []
    while not stop_event.is_set():
        try:
            out = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu",
                    "--format=csv,noheader,nounits",
                ],
                capture_output=True,
                text=True,
                timeout=5,
            ).stdout.strip()
            if out.isdigit():
                samples.append(int(out))
        except (FileNotFoundError, subprocess.TimeoutExpired):
            pass
        time.sleep(interval_s)
    return samples


def process_uses_cuda(pid: int) -> bool:
    """Second, independent signal for GPU use: checks whether the process
    has any CUDA/NVIDIA library mapped into its address space. Disagreement
    with the utilization sampler should be reported as inconclusive, not
    resolved by silently picking one signal over the other."""
    try:
        maps = Path(f"/proc/{pid}/maps").read_text()
        return "cuda" in maps.lower() or "nvidia" in maps.lower()
    except (FileNotFoundError, PermissionError):
        return False


def timed_ingest(upsert_fn, documents: list[str], server_pid: int | None = None) -> dict:
    """Times upsert_fn(documents) -- a caller-supplied closure over
    whichever endpoint validated (Qdrant-dialect or OpenSearch-autoembed)
    -- while concurrently sampling GPU utilization, so "did this use the
    GPU" is an empirical result rather than an assumption from the runtime
    type alone."""
    stop_event = threading.Event()
    samples: list[int] = []

    def _collect():
        samples.extend(sample_gpu_utilization(stop_event))

    sampler = threading.Thread(target=_collect)
    sampler.start()

    start = time.perf_counter()
    upsert_fn(documents)
    elapsed = time.perf_counter() - start

    stop_event.set()
    sampler.join()

    max_util = max(samples, default=0)
    mean_util = sum(samples) / len(samples) if samples else 0.0
    cuda_loaded = process_uses_cuda(server_pid) if server_pid else None

    return {
        "documents_ingested": len(documents),
        "wall_clock_seconds": elapsed,
        "documents_per_second": len(documents) / elapsed if elapsed > 0 else 0.0,
        "gpu_utilization_max_pct": max_util,
        "gpu_utilization_mean_pct": mean_util,
        "gpu_utilization_samples": len(samples),
        "process_maps_show_cuda": cuda_loaded,
        "gpu_acceleration_observed": bool(max_util > 5 or cuda_loaded),
    }

## 4. Dataset: kubernetes/website docs (CC-BY-4.0)

The base repo's own `sample_corpus/` is 6 files -- too small for a GPU-vs-CPU
comparison to mean anything; the gap would be dominated by fixed overhead
(model load, process startup) rather than embedding throughput.

Instead we use the **Kubernetes documentation** (`kubernetes/website`,
`content/en/docs/` subtree), pulled via a partial/sparse clone so only that
subtree's blobs are fetched:

- **License:** CC-BY-4.0 (verified at
  `github.com/kubernetes/website/blob/master/LICENSE`) -- safe to publish
  benchmark artifacts derived from it, with attribution.
- **Real and large:** thousands of real technical-writing markdown files,
  deeply nested (`concepts/`, `tasks/`, `tutorials/`, `reference/`,
  `setup/`, each with further subdirectories) -- a genuine exercise of the
  non-recursive-per-directory chunker, not a flat directory that would
  sidestep it.
- **Known quirk, not fixed:** these files use Hugo front matter (`---
  ... ---`) and Hugo shortcodes (double-curly note/warning blocks).
  The base repo's chunker has no special handling for either (it is
  protected and unmodified) -- expect minor text noise in some chunks.
  This is called out here so it isn't mistaken for a bug.

In [ ]:
!mkdir -p /content/corpus_src
!git clone --depth 1 --filter=blob:none --sparse https://github.com/kubernetes/website.git /content/corpus_src/k8s-website
%cd /content/corpus_src/k8s-website
!git sparse-checkout set content/en/docs
%cd /content/local-inference-bench

In [ ]:
import subprocess

sha = subprocess.run(
    ["git", "-C", "/content/corpus_src/k8s-website", "rev-parse", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()
license_text = open("/content/corpus_src/k8s-website/LICENSE").read()[:400]

corpus_provenance = {
    "repo": "https://github.com/kubernetes/website",
    "subtree": "content/en/docs",
    "commit": sha,
    "license": "CC-BY-4.0",
    "license_header": license_text,
    "retrieved_utc": env["run_started_utc"],
}
print(json.dumps(corpus_provenance, indent=2))

In [ ]:
from pathlib import Path
from multi_dir_corpus import load_corpus_recursive

MAX_CHUNKS = 6000  # explicit, reproducible cap -- keeps runtime predictable
                    # regardless of how large the docs tree is on a given day

chunks, leaf_dirs = load_corpus_recursive(
    root=Path("/content/corpus_src/k8s-website/content/en/docs"),
    max_chunks=MAX_CHUNKS,
    seed=42,
)
documents = [c.text for c in chunks]

print(f"leaf directories with .md files: {len(leaf_dirs)}")
print(f"chunks: {len(chunks)}  (cap was {MAX_CHUNKS})")
print(f"total words: {sum(c.word_count for c in chunks)}")
print(f"source files represented: {len(set(c.source_file for c in chunks))}")

In [ ]:
from tokens import count_tokens, TOKENIZER_MODEL

num_tokens = count_tokens(documents)
token_lengths = None  # computed lazily below only if needed
print(f"total tokens ({TOKENIZER_MODEL} / cl100k_base): {num_tokens}")
print(f"avg tokens/chunk: {num_tokens / len(documents):.1f}")

# FastEmbed's bge-small-en-v1.5 truncates at 512 tokens per input. Report
# the corpus's own token-length distribution so truncation impact (if
# any) is visible rather than assumed away.
import tiktoken
enc = tiktoken.encoding_for_model(TOKENIZER_MODEL)
per_doc_tokens = [len(enc.encode(d)) for d in documents]
over_512 = sum(1 for t in per_doc_tokens if t > 512)
print(f"chunks over the model's 512-token input limit: {over_512} "
      f"({100 * over_512 / len(per_doc_tokens):.1f}% of {len(per_doc_tokens)})")
print(f"max chunk token length: {max(per_doc_tokens)}")

## 5. CPU baseline (unmodified `run_local_benchmark.py`)

This drives the base repo's own, unmodified script as a subprocess against
every leaf directory just discovered -- guaranteeing this CPU number is
methodology-identical to the rest of the repo, just measured on Colab's
CPU instead of the Mac/AWS-cited one.

In [ ]:
import subprocess

Path("results").mkdir(exist_ok=True)

dirs_args = []
for d in leaf_dirs:
    dirs_args += ["--corpus-dir", str(d)]

subprocess.run(
    ["python", "run_local_benchmark.py", *dirs_args,
     "--out", "results/results_cpu_colab.json"],
    check=True,
)

In [ ]:
cpu_result = json.load(open("results/results_cpu_colab.json"))
print(json.dumps(cpu_result, indent=2))

## 6. GPU FastEmbed leg

**Package conflict warning:** `fastembed` (plain, CPU) and `fastembed-gpu`
ship different, same-named native ONNX Runtime libraries. Installing both
in the same environment causes one to silently overwrite the other's
libraries (a documented failure mode -- see `qdrant/fastembed#608`), which
would produce a "GPU" run that's silently actually CPU. So this cell
**uninstalls** the plain packages installed by `requirements.txt` in
section 2 before installing the GPU stack.

This leg is skipped entirely (not faked) if `HAS_GPU` was `False` above.

In [ ]:
if HAS_GPU:
    !pip uninstall -y onnxruntime fastembed -q
    !pip install -q fastembed-gpu==0.8.0
    !pip install -q "onnxruntime-gpu==1.18.0" \
        -i https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
else:
    print("HAS_GPU is False -- skipping GPU package install.")

In [ ]:
gpu_result = None

if HAS_GPU:
    import time
    from fastembed import TextEmbedding

    model = TextEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
    )

    # Hard verification: check the ACTIVE onnxruntime providers, not just
    # what was requested -- a silent CPU fallback must not be reported as
    # a GPU number.
    active_providers = model.model.model.get_providers()
    gpu_active = "CUDAExecutionProvider" in active_providers
    print("onnxruntime active providers:", active_providers)
    if not gpu_active:
        print("WARNING: CUDAExecutionProvider not active -- this run is "
              "CPU, not GPU. It will be tagged accordingly, not reported "
              "as a GPU result.")

    t_load_start = time.perf_counter()
    _ = list(model.embed(["warmup"]))  # exclude first-call CUDA context init from timing
    model_load_seconds = time.perf_counter() - t_load_start

    t0 = time.perf_counter()
    embeddings = list(model.embed(documents, batch_size=256))
    wall = time.perf_counter() - t0

    embedding_dim = len(embeddings[0]) if embeddings else 0
    docs_per_sec = len(documents) / wall

    source_tag = "measured" if gpu_active else "measured (invalid: no GPU acceleration detected)"
    print(f"[{source_tag}] {len(documents)} docs in {wall:.2f}s -> {docs_per_sec:.1f} docs/sec")

    gpu_result = {
        "corpus": {
            "num_chunks": len(chunks),
            "total_words": sum(c.word_count for c in chunks),
            "num_tokens": num_tokens,
            "tokenizer_model": TOKENIZER_MODEL,
            "num_source_files": len(set(c.source_file for c in chunks)),
            "corpus_dirs": [str(d) for d in leaf_dirs],
        },
        "benchmark": {
            "model_name": "BAAI/bge-small-en-v1.5",
            "documents_embedded": len(documents),
            "embedding_dim": embedding_dim,
            "wall_clock_seconds": wall,
            "documents_per_second": docs_per_sec,
            "model_load_seconds": model_load_seconds,
            "fastembed_batch_size": 256,
            "platform": env["platform"],
            "python_version": env["python_version"],
        },
        "gpu_info": {
            "onnxruntime_active_providers": active_providers,
            "gpu_active": gpu_active,
            "nvidia_smi": env["nvidia_smi"],
        },
        "source": source_tag,
    }
    json.dump(gpu_result, open("results/results_gpu.json", "w"), indent=2)
    print("Wrote results/results_gpu.json")
else:
    print("HAS_GPU is False -- GPU FastEmbed leg skipped entirely, no result written.")

## 7. Local/embedded Qdrant leg

Mirrors the base repo's own `push_to_qdrant.py` pattern (an in-process
`QdrantClient`, no server) but uses **on-disk** storage rather than
`:memory:` -- the closer analog to a real self-hosted deployment, since it
produces a durable index whose size can actually be reported, and it
exercises real disk I/O rather than only RAM.

Requires the GPU-embedded vectors from section 6; skipped if that leg was
skipped.

In [ ]:
qdrant_result = None

if gpu_result is not None:
    import os
    import time
    import psutil
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, PointStruct, VectorParams

    os.makedirs("/content/qdrant_local_data", exist_ok=True)
    client = QdrantClient(path="/content/qdrant_local_data")

    if client.collection_exists("k8s_docs"):
        client.delete_collection("k8s_docs")
    client.create_collection(
        collection_name="k8s_docs",
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )

    proc = psutil.Process(os.getpid())
    rss_before = proc.memory_info().rss

    t0 = time.perf_counter()
    points = [
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload={"source_file": chunks[i].source_file, "chunk_index": chunks[i].chunk_index},
        )
        for i in range(len(chunks))
    ]
    client.upsert(collection_name="k8s_docs", points=points, wait=True)
    upsert_wall = time.perf_counter() - t0
    rss_after = proc.memory_info().rss

    disk_bytes = int(
        subprocess.run(["du", "-sb", "/content/qdrant_local_data"], capture_output=True, text=True)
        .stdout.split()[0]
    )
    points_per_sec = len(points) / upsert_wall

    # A handful of timed searches, since a vector store's job is queries,
    # not just ingest.
    query_vec = embeddings[0].tolist()
    t0 = time.perf_counter()
    for _ in range(20):
        client.query_points(collection_name="k8s_docs", query=query_vec, limit=5)
    query_wall = (time.perf_counter() - t0) / 20

    print(f"upserted {len(points)} points in {upsert_wall:.2f}s -> {points_per_sec:.1f} points/sec")
    print(f"RSS delta: {(rss_after - rss_before) / 1e6:.1f} MB")
    print(f"on-disk index size: {disk_bytes / 1e6:.1f} MB")
    print(f"mean query latency (20 queries, top-5): {query_wall * 1000:.1f} ms")

    qdrant_result = {
        "points_upserted": len(points),
        "upsert_wall_clock_seconds": upsert_wall,
        "points_per_second": points_per_sec,
        "rss_delta_mb": (rss_after - rss_before) / 1e6,
        "disk_size_mb": disk_bytes / 1e6,
        "mean_query_latency_ms": query_wall * 1000,
        "source": "measured",
    }
    json.dump(qdrant_result, open("results/results_qdrant_local.json", "w"), indent=2)
    print("Wrote results/results_qdrant_local.json")
else:
    print("GPU leg was skipped -- local Qdrant leg skipped too (needs GPU-embedded vectors).")

## 8. SearchAI Index Server: install + probe

**Two real, up-front risks, stated before running anything:**

1. **Systemd-in-Colab risk.** The official one-line installer's last step
   sets up a systemd service; Colab's VM has no full init system. This
   notebook's primary path downloads the same release tarball the
   installer fetches and runs `./run.sh` **in the foreground** instead
   (a documented manual alternative), which needs no systemd. The full
   installer is only attempted as a fallback, and is allowed to fail
   loudly rather than being retried silently.
2. **The Qdrant-dialect plain-text-upsert premise is unverified.** No
   published SearchBlox documentation shows a vector-less upsert body for
   the Qdrant dialect. If the server rejects it, that is reported as an
   explicit, valid result with the exact request/response -- not routed
   around by silently switching to a different API.

In [ ]:
import subprocess
from pathlib import Path
import searchai_client as sc

bundle_dir = sc.download_and_unpack(version="1.3.0", arch="amd64", work_dir=Path("/content"))
print("unpacked to:", bundle_dir)
!ls "{bundle_dir}"

In [ ]:
log_path = Path("/content/searchai_run.log")
searchai_proc = sc.start_foreground(bundle_dir, log_path)
print("started run.sh, pid:", searchai_proc.pid)

searchai_healthy = sc.wait_for_health(port=9200, proc=searchai_proc, log_path=log_path, timeout_s=150)
print("SearchAI foreground path healthy:", searchai_healthy)

In [ ]:
# Fallback ONLY if the foreground path failed. Allowed to fail loudly --
# no retry loop, no silent workaround.
if not searchai_healthy:
    print("Foreground run.sh path failed. Attempting the full installer as")
    print("a documented fallback (systemd-based -- may not work in Colab).")
    result = subprocess.run(
        "curl -fsSL https://index-server.searchblox.com/install | sudo PORT=9200 VERSION=1.3.0 bash",
        shell=True, capture_output=True, text=True,
    )
    print("installer exit code:", result.returncode)
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
    status = subprocess.run(["systemctl", "status", "searchai-index"], capture_output=True, text=True)
    print(status.stdout, status.stderr)
    searchai_healthy = sc.wait_for_health(port=9200, timeout_s=60)
    print("SearchAI healthy after installer fallback:", searchai_healthy)

In [ ]:
api_key = sc.find_admin_api_key(log_path)
if api_key:
    print(f"found admin API key: {api_key[:4]}...{api_key[-4:]} (redacted)")
else:
    print("no admin API key found in the launch log -- trying unauthenticated "
          "requests (auth is documented as off by default).")

if searchai_healthy:
    version_probe = sc.probe_version(port=9200, api_key=api_key)
    print(version_probe.status_code, version_probe.response_body[:1000])

In [ ]:
if searchai_healthy:
    sc.enable_qdrant_dialect(prefix=bundle_dir, qdrant_port=6333)
    searchai_proc.terminate()
    searchai_proc.wait(timeout=30)
    searchai_proc = sc.start_foreground(bundle_dir, log_path)
    print("restarted run.sh (Qdrant dialect enabled), pid:", searchai_proc.pid)

    searchai_healthy = sc.wait_for_health(port=9200, proc=searchai_proc, log_path=log_path, timeout_s=150)
    print("healthy after restart:", searchai_healthy)

    import socket
    try:
        with socket.create_connection(("localhost", 6333), timeout=5):
            qdrant_dialect_up = True
    except OSError:
        qdrant_dialect_up = False
    print("Qdrant dialect listener (port 6333) reachable:", qdrant_dialect_up)

In [ ]:
searchai_result = {"source": "measured (not attempted: server did not become healthy)"}

if searchai_healthy and qdrant_dialect_up:
    # Probe #1: known-good collection create (client-supplied vector config).
    create_probe = sc.probe_create_qdrant_collection("k8s_probe", dim=384, qdrant_port=6333)
    print("collection create:", create_probe.status_code, create_probe.response_body[:500])

    # Probe #2: THE critical test -- a vector-less, text-only upsert.
    text_upsert_probe = sc.probe_plaintext_upsert("k8s_probe", chunks[0].text, qdrant_port=6333)
    print("plain-text upsert:", text_upsert_probe.status_code, text_upsert_probe.response_body[:1000])
    SEARCHAI_TEXT_UPSERT_SUPPORTED = text_upsert_probe.ok

    # Probe #3: the alternate, independently-documented OpenSearch
    # knn_vector + "embed" mapping path. A pass here does NOT validate the
    # Qdrant-dialect premise above -- reported separately, always.
    os_create, os_index = sc.probe_opensearch_autoembed(
        "k8s_probe_os", chunks[0].text, model="bge-small-en-v1.5", dim=384, port=9200
    )
    print("OpenSearch auto-embed mapping create:", os_create.status_code, os_create.response_body[:500])
    print("OpenSearch auto-embed doc index:", os_index.status_code, os_index.response_body[:500])
    SEARCHAI_OPENSEARCH_AUTOEMBED_SUPPORTED = os_index.ok

    if not SEARCHAI_TEXT_UPSERT_SUPPORTED:
        print("=" * 70)
        print("HARD STOP (Qdrant-dialect leg): plain-text upsert was REJECTED.")
        print("This falsifies the \'server-side embedding via Qdrant dialect\' premise.")
        print("Exact status:", text_upsert_probe.status_code)
        print("Exact response:", text_upsert_probe.response_body)
        print("=" * 70)

    searchai_result = {
        "qdrant_dialect_reachable": qdrant_dialect_up,
        "qdrant_collection_create": {
            "status_code": create_probe.status_code,
            "response": create_probe.response_body[:2000],
        },
        "qdrant_plaintext_upsert_supported": SEARCHAI_TEXT_UPSERT_SUPPORTED,
        "qdrant_plaintext_upsert_probe": {
            "request": text_upsert_probe.request,
            "status_code": text_upsert_probe.status_code,
            "response": text_upsert_probe.response_body[:2000],
        },
        "opensearch_autoembed_supported": SEARCHAI_OPENSEARCH_AUTOEMBED_SUPPORTED,
        "opensearch_autoembed_probe": {
            "create_status_code": os_create.status_code,
            "create_response": os_create.response_body[:2000],
            "index_status_code": os_index.status_code,
            "index_response": os_index.response_body[:2000],
        },
        "source": "measured" if (SEARCHAI_TEXT_UPSERT_SUPPORTED or SEARCHAI_OPENSEARCH_AUTOEMBED_SUPPORTED)
                   else "measured (not supported)",
    }
else:
    print("Server never became healthy with the Qdrant dialect reachable -- "
          "skipping ingest probes entirely. See searchai_result for the reason.")

## 9. SearchAI ingest + GPU-observation (only if a path validated above)

Times a full-corpus ingest through whichever endpoint validated (Qdrant
dialect or OpenSearch auto-embed), with a concurrent `nvidia-smi`
utilization sampler and a `/proc/<pid>/maps` CUDA-library check, so "did
SearchAI use the GPU" is answered empirically rather than assumed from
having a GPU runtime attached.

In [ ]:
if searchai_result.get("qdrant_plaintext_upsert_supported") or searchai_result.get("opensearch_autoembed_supported"):
    import requests

    def qdrant_text_upsert_fn(docs):
        points = [{"id": i, "payload": {"body": d}} for i, d in enumerate(docs)]
        requests.put(
            "http://localhost:6333/collections/k8s_full/points",
            json={"points": points}, timeout=120,
        )

    def opensearch_autoembed_upsert_fn(docs):
        for i, d in enumerate(docs):
            requests.post(
                f"http://localhost:9200/k8s_full_os/_doc/{i}",
                json={"body": d}, timeout=30,
            )

    if searchai_result.get("qdrant_plaintext_upsert_supported"):
        sc.probe_create_qdrant_collection("k8s_full", dim=384, qdrant_port=6333)
        ingest_fn, ingest_label = qdrant_text_upsert_fn, "qdrant_dialect_plaintext"
    else:
        sc.probe_opensearch_autoembed("k8s_full_os", "warmup", port=9200)  # create mapping once
        ingest_fn, ingest_label = opensearch_autoembed_upsert_fn, "opensearch_autoembed"

    ingest_stats = sc.timed_ingest(ingest_fn, documents, server_pid=searchai_proc.pid)
    ingest_stats["ingest_path"] = ingest_label
    print(json.dumps(ingest_stats, indent=2))

    verdict = "CONFIRMED" if ingest_stats["gpu_acceleration_observed"] else "NOT observed (treat as CPU-bound)"
    print(f"\nGPU acceleration during SearchAI ingest: {verdict}")

    searchai_result["ingest"] = ingest_stats
else:
    print("No validated ingest path -- skipping full-corpus SearchAI ingest.")

json.dump(searchai_result, open("results/results_searchai.json", "w"), indent=2)
print("\nWrote results/results_searchai.json")

## 10. Combined results + comparison table

In [ ]:
from pricing import OPENAI_TEXT_EMBEDDING_3_SMALL, AWS_C7G_XLARGE

combined = {
    "colab_environment": {
        "nvidia_smi": env["nvidia_smi"],
        "has_gpu": HAS_GPU,
        "platform": env["platform"],
        "run_started_utc": env["run_started_utc"],
    },
    "corpus_provenance": corpus_provenance,
    "legs": {
        "cpu_baseline": cpu_result,
        "gpu_fastembed": gpu_result,
        "qdrant_local": qdrant_result,
        "searchai": searchai_result,
    },
    "cited": {
        "openai_text_embedding_3_small": {
            "usd": OPENAI_TEXT_EMBEDDING_3_SMALL.usd,
            "unit": OPENAI_TEXT_EMBEDDING_3_SMALL.unit,
            "source": OPENAI_TEXT_EMBEDDING_3_SMALL.source,
            "checked_date": OPENAI_TEXT_EMBEDDING_3_SMALL.checked_date,
        },
        "aws_c7g_xlarge": {
            "usd": AWS_C7G_XLARGE.usd,
            "unit": AWS_C7G_XLARGE.unit,
            "source": AWS_C7G_XLARGE.source,
            "checked_date": AWS_C7G_XLARGE.checked_date,
        },
    },
}
json.dump(combined, open("results/results_colab_combined.json", "w"), indent=2)
print("Wrote results/results_colab_combined.json")

In [ ]:
print("Sanity check: the unmodified cost_comparison.py still runs against")
print("the new (larger, real) corpus's CPU-baseline results file:\n")
!python cost_comparison.py --results results/results_cpu_colab.json

In [ ]:
gpu_name = env["nvidia_smi"].split(",")[0] if HAS_GPU else "none"

print("=" * 72)
print(f"MEASURED: Colab CPU baseline (bge-small-en-v1.5, FastEmbed, {cpu_result['benchmark']['cpu_cores']} cores)")
print("=" * 72)
b = cpu_result["benchmark"]
print(f"{cpu_result['corpus']['num_chunks']} chunks, {cpu_result['corpus']['num_tokens']} tokens, "
      f"{b['wall_clock_seconds']:.2f}s wall -> {b['documents_per_second']:.1f} docs/sec   [source: measured]")
print("NOTE: this is Colab's own vCPU, not the AWS c7g.xlarge cited below -- a new data point, not a substitute.")
print()

print("=" * 72)
print(f"MEASURED: Colab GPU ({gpu_name}, bge-small-en-v1.5, fastembed-gpu)")
print("=" * 72)
if gpu_result is not None:
    gb = gpu_result["benchmark"]
    print(f"{gpu_result['corpus']['num_chunks']} chunks, {gb['wall_clock_seconds']:.2f}s wall -> "
          f"{gb['documents_per_second']:.1f} docs/sec   [source: {gpu_result['source']}]")
    print(f"onnxruntime active providers: {gpu_result['gpu_info']['onnxruntime_active_providers']}")
else:
    print("SKIPPED -- no GPU runtime attached this session.")
print()

print("=" * 72)
print("MEASURED: Local Qdrant (on-disk, same GPU-embedded vectors)")
print("=" * 72)
if qdrant_result is not None:
    print(f"{qdrant_result['points_upserted']} points upserted in {qdrant_result['upsert_wall_clock_seconds']:.2f}s "
          f"-> {qdrant_result['points_per_second']:.1f} pts/sec, "
          f"disk size {qdrant_result['disk_size_mb']:.1f} MB, "
          f"RSS delta {qdrant_result['rss_delta_mb']:.1f} MB, "
          f"mean query {qdrant_result['mean_query_latency_ms']:.1f} ms   [source: measured]")
else:
    print("SKIPPED -- depends on the GPU leg, which was skipped.")
print()

print("=" * 72)
print("MEASURED: SearchAI Index Server v1.3.0 (Colab, foreground run.sh)")
print("=" * 72)
print(f"Qdrant-dialect plain-text upsert supported: "
      f"{searchai_result.get('qdrant_plaintext_upsert_supported', 'not reached')}")
print(f"OpenSearch knn_vector auto-embed supported: "
      f"{searchai_result.get('opensearch_autoembed_supported', 'not reached')}")
if "ingest" in searchai_result:
    ing = searchai_result["ingest"]
    print(f"Ingest ({ing['ingest_path']}): {ing['documents_ingested']} docs in "
          f"{ing['wall_clock_seconds']:.2f}s -> {ing['documents_per_second']:.1f} docs/sec")
    print(f"GPU utilization during ingest: max {ing['gpu_utilization_max_pct']}% "
          f"mean {ing['gpu_utilization_mean_pct']:.1f}%  "
          f"CUDA loaded in process maps: {ing['process_maps_show_cuda']}")
    print(f"GPU acceleration: {'CONFIRMED' if ing['gpu_acceleration_observed'] else 'NOT observed (CPU-bound)'}")
print(f"[source: {searchai_result['source']}]")
print()

print("=" * 72)
print("CITED (unchanged from the base repo's pricing.py)")
print("=" * 72)
print(f"OpenAI text-embedding-3-small: ${OPENAI_TEXT_EMBEDDING_3_SMALL.usd} "
      f"{OPENAI_TEXT_EMBEDDING_3_SMALL.unit} ({OPENAI_TEXT_EMBEDDING_3_SMALL.source}, "
      f"checked {OPENAI_TEXT_EMBEDDING_3_SMALL.checked_date})")
print(f"AWS c7g.xlarge on-demand: ${AWS_C7G_XLARGE.usd} {AWS_C7G_XLARGE.unit} "
      f"({AWS_C7G_XLARGE.source}, checked {AWS_C7G_XLARGE.checked_date})")

## 11. Download results before the runtime recycles

Colab sessions are ephemeral -- run this now.

In [ ]:
import shutil
shutil.make_archive("/content/colab_benchmark_results", "zip", "results")
try:
    from google.colab import files
    files.download("/content/colab_benchmark_results.zip")
except ImportError:
    print("Not running in Colab -- results are at /content/colab_benchmark_results.zip")